In [ ]:
# ==========================================
# STEP 1: Setup & Authentication
# ==========================================
!pip install -q -U transformers peft datasets trl accelerate
from huggingface_hub import notebook_login
notebook_login() # Requires 'Read' access token

In [ ]:
# ==========================================
# STEP 2: Load Model & Tokenizer (Raw Base Model)
# ==========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load the raw, un-wrapped base model in high precision
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

In [ ]:
# ==========================================
# STEP 3: Dataset & Formatting
# ==========================================
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

# Split off 5% for validation
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

def format_instruction(sample):
    return f"### Instruction:\n{sample['instruction']}\n\n### Context:\n{sample['context']}\n\n### Response:\n{sample['response']}"

In [ ]:
# ==========================================
# STEP 4: Configuration
# ==========================================
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

training_args = TrainingArguments(
    output_dir="./gemma-study",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=1000,
    logging_steps=10,
    bf16=True,
    report_to="none",
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=4,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    args=training_args,
    formatting_func=format_instruction,
)

In [ ]:
# ==========================================
# STEP 5: Train and Test
# ==========================================
print("Starting training...")
trainer.train()

In [ ]:
# ==========================================
# STEP 5.5: Plot loss curves  (run after training)
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(log_df.dropna(subset=["loss"])["step"],
        log_df.dropna(subset=["loss"])["loss"],
        label="Train loss")
ax.plot(log_df.dropna(subset=["eval_loss"])["step"],
        log_df.dropna(subset=["eval_loss"])["eval_loss"],
        label="Val loss", linestyle="--")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Gemma-2 2B — LoRA fine-tuning on Dolly-15k")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def compare_models(prompt_text):
    prompt = f"### Instruction:\n{prompt_text}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print(f"=== PROMPT: {prompt_text} ===\n")

    # 1. Test the ORIGINAL Base Model (by disabling adapters)
    # Notice we are using trainer.model here!
    with trainer.model.disable_adapter():
        with torch.no_grad():
            base_outputs = trainer.model.generate(**inputs, max_new_tokens=100)
    print("🔴 ORIGINAL MODEL (Base):")
    print(tokenizer.decode(base_outputs[0], skip_special_tokens=True).replace(prompt, ""))
    print("-" * 40)

    # 2. Test the NEW Finetuned Model (adapters automatically active)
    # Notice we are using trainer.model here too!
    with torch.no_grad():
        finetuned_outputs = trainer.model.generate(**inputs, max_new_tokens=100)
    print("🟢 FINETUNED MODEL (LoRA):")
    print(tokenizer.decode(finetuned_outputs[0], skip_special_tokens=True).replace(prompt, ""))
    print("=" * 60 + "\n")

# Run the tests!
compare_models("What is the capital of France?")
compare_models("Explain gravity to a 5-year old.")

In [ ]:
from google.colab import drive
import os

# 1. Connect to Google Drive
drive.mount('/content/drive')

# 2. Define the save path
save_path = "/content/drive/MyDrive/gemma-2-dolly-lora"

# 3. Save the LoRA adapters and the tokenizer
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model successfully saved to {save_path}")

# Ladataan malli ja koulutetut LoRA adapterit

In [ ]:
!pip install -q -U transformers peft accelerate
from huggingface_hub import notebook_login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# You need to login again to access the base Gemma 2 model
notebook_login()

# 1. Setup paths
base_model_id = "google/gemma-2-2b"
adapter_path = "/content/drive/MyDrive/gemma-2-dolly-lora"

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# 3. Load the Base Model (in BF16, just like training)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 4. Snap the LoRA adapters onto the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
print("Model successfully loaded and ready for chat!")

# --- You can now test it immediately ---
prompt = "### Instruction:\nWhy is the sky blue?\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))